# Flood EO Indicators Extraction

<div style="border: 2px solid #4A90E2; border-radius: 8px; padding: 16px; background: #F7FBFF; font-size: 15px; line-height: 1.5;">
<h3 style="margin-top:0;">What this notebook does</h3>
<p>This notebook is the flood-specific EO extraction step in the EO2Explain pipeline. It takes already downloaded before/after multispectral imagery for each event and converts it into symbolic indicators that can later be used by the MAS reasoning layer.</p>
<h4 style="margin-bottom:4px;">Main purpose</h4>
<ul style="margin-top:0;">
<li>compare pre-event and post-event imagery for each flood case</li>
<li>derive compact flood-related indicators from spectral data</li>
<li>produce structured outputs for reasoning, analysis, and reporting</li>
</ul>
<h4 style="margin-bottom:4px;">Inputs</h4>
<ul style="margin-top:0;">
<li>local event folders with <code>before_flooding</code> and <code>after_flooding</code> imagery</li>
<li>at least bands <code>B03</code> and <code>B08</code> for NDWI-based water detection</li>
<li>band files whose names contain the band identifiers</li>
</ul>
<h4 style="margin-bottom:4px;">What is computed</h4>
<ul style="margin-top:0;">
<li><b>NDWI</b> before and after</li>
<li><b>water area percentage</b> before and after</li>
<li><b>water increase percentage</b></li>
<li><b>newly flooded area percentage</b></li>
</ul>
<h4 style="margin-bottom:4px;">Outputs</h4>
<ul style="margin-top:0;">
<li>a summary CSV in <code>data/processed/indicators/</code></li>
<li>optional diagnostic figures for selected events</li>
<li>event-level metrics that feed the downstream symbolic hazard reasoning layer</li>
</ul>
<h4 style="margin-bottom:4px;">Role in EO2Explain</h4>
<p style="margin-bottom:0;">This notebook belongs to the EO processing layer. It does not perform hazard reasoning itself; instead, it prepares the measurable evidence that the MAS later interprets symbolically.</p>
</div>

This notebook scans a folder structured like:

```text
data/raw/floods/
├── Event_1/
│   ├── before_flooding/
│   │   ├── ...B03...(Raw).tiff
│   │   ├── ...B08...(Raw).tiff
│   │   └── ...B11...(Raw).tiff   # optional, not required for NDWI
│   └── after_flooding/
│       ├── ...B03...(Raw).tiff
│       ├── ...B08...(Raw).tiff
│       └── ...B11...(Raw).tiff
└── Event_2/
    └── ...
```


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 6)
pd.set_option("display.max_columns", None)

ROOT_DIR = Path("../../data/raw/floods")

BEFORE_FOLDER = "before_flooding"
AFTER_FOLDER = "after_flooding"

# Threshold for water mask from NDWI
WATER_NDWI_THRESHOLD = 0.0

ROOT_DIR


In [ ]:
def find_band_file(folder: Path, band: str):
    candidates = sorted(folder.glob("*.tif")) + sorted(folder.glob("*.tiff"))
    for f in candidates:
        if re.search(fr"[_-]{band}([_-]|\b)", f.name, flags=re.IGNORECASE) or band.lower() in f.name.lower():
            return f
    raise FileNotFoundError(f"Band {band} not found in {folder}")

def read_band(path: Path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    arr = np.where(np.isfinite(arr), arr, np.nan)
    return arr, profile

def safe_index(num, den):
    out = np.divide(num, den, out=np.full_like(num, np.nan, dtype="float32"), where=np.abs(den) > 1e-6)
    return out


In [ ]:
def compute_flood_metrics(event_dir: Path):
    before_dir = event_dir / BEFORE_FOLDER
    after_dir = event_dir / AFTER_FOLDER

    b3_before_path = find_band_file(before_dir, "B03")
    b8_before_path = find_band_file(before_dir, "B08")
    b3_after_path = find_band_file(after_dir, "B03")
    b8_after_path = find_band_file(after_dir, "B08")

    b3_before, profile = read_band(b3_before_path)
    b8_before, _ = read_band(b8_before_path)
    b3_after, _ = read_band(b3_after_path)
    b8_after, _ = read_band(b8_after_path)

    ndwi_before = safe_index(b3_before - b8_before, b3_before + b8_before)
    ndwi_after = safe_index(b3_after - b8_after, b3_after + b8_after)

    valid = np.isfinite(ndwi_before) & np.isfinite(ndwi_after)
    water_before = valid & (ndwi_before > WATER_NDWI_THRESHOLD)
    water_after = valid & (ndwi_after > WATER_NDWI_THRESHOLD)
    new_flood = valid & (~water_before) & water_after

    valid_pixels = int(valid.sum())
    water_before_pixels = int(water_before.sum())
    water_after_pixels = int(water_after.sum())
    new_flood_pixels = int(new_flood.sum())

    metrics = {
        "event": event_dir.name,
        "before_folder": str(before_dir),
        "after_folder": str(after_dir),
        "mean_ndwi_before": float(np.nanmean(ndwi_before)),
        "mean_ndwi_after": float(np.nanmean(ndwi_after)),
        "ndwi_change": float(np.nanmean(ndwi_after) - np.nanmean(ndwi_before)),
        "water_area_before_pct": 100.0 * water_before_pixels / valid_pixels if valid_pixels else np.nan,
        "water_area_after_pct": 100.0 * water_after_pixels / valid_pixels if valid_pixels else np.nan,
        "water_increase_pct_points": (
            100.0 * water_after_pixels / valid_pixels - 100.0 * water_before_pixels / valid_pixels
        ) if valid_pixels else np.nan,
        "newly_flooded_area_pct": 100.0 * new_flood_pixels / valid_pixels if valid_pixels else np.nan,
        "valid_pixels": valid_pixels,
        "water_before_pixels": water_before_pixels,
        "water_after_pixels": water_after_pixels,
        "new_flood_pixels": new_flood_pixels,
    }

    arrays = {
        "ndwi_before": ndwi_before,
        "ndwi_after": ndwi_after,
        "water_before": water_before,
        "water_after": water_after,
        "new_flood": new_flood,
        "profile": profile,
    }

    return metrics, arrays


In [ ]:
event_dirs = sorted([p for p in ROOT_DIR.iterdir() if p.is_dir()])
print("Events found:", [p.name for p in event_dirs])

results = []
arrays_by_event = {}

for event_dir in event_dirs:
    try:
        metrics, arrays = compute_flood_metrics(event_dir)
        results.append(metrics)
        arrays_by_event[event_dir.name] = arrays
        print(f"Processed: {event_dir.name}")
    except Exception as e:
        print(f"Skipped {event_dir.name}: {e}")

flood_df = pd.DataFrame(results).sort_values("event").reset_index(drop=True)
flood_df


In [ ]:
output_csv = Path("../../data/processed/indicators/flood_eo_indicators_summary.csv")
output_csv.parent.mkdir(parents=True, exist_ok=True)
flood_df.to_csv(output_csv, index=False)
print(f"Saved summary to: {output_csv}")


In [ ]:
event_to_plot = flood_df.loc[1, "event"] if len(flood_df) else None
event_to_plot


In [ ]:
def plot_flood_report_figure(arrs, event_name, ndwi_threshold=WATER_NDWI_THRESHOLD):
    """
    Parameters
    ----------
    arrs : dict
        Dictionary containing:
        - ndwi_before
        - ndwi_after
        - water_before
        - water_after
        - new_flood
    event_name : str
        Event label to use in titles.
    ndwi_threshold : float
        Threshold used to derive water masks; included in the figure caption text if needed.
    """

    ndwi_before = arrs["ndwi_before"]
    ndwi_after = arrs["ndwi_after"]
    water_before = arrs["water_before"]
    water_after = arrs["water_after"]
    new_flood = arrs["new_flood"]

    ndwi_change = np.where(
        np.isfinite(ndwi_before) & np.isfinite(ndwi_after),
        ndwi_after - ndwi_before,
        np.nan
    )

    # Robust symmetric range for change plot
    finite_change = ndwi_change[np.isfinite(ndwi_change)]
    if finite_change.size > 0:
        vmax_change = np.nanpercentile(np.abs(finite_change), 98)
        vmax_change = max(vmax_change, 0.05)
    else:
        vmax_change = 0.2

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
    axes = axes.ravel()

    # 1. NDWI before
    im0 = axes[0].imshow(ndwi_before, cmap="Blues", vmin=-1, vmax=1)
    axes[0].set_title(f"{event_name} - NDWI before")
    axes[0].axis("off")
    cbar0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    cbar0.set_label("NDWI")

    # 2. NDWI after
    im1 = axes[1].imshow(ndwi_after, cmap="Blues", vmin=-1, vmax=1)
    axes[1].set_title(f"{event_name} - NDWI after")
    axes[1].axis("off")
    cbar1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    cbar1.set_label("NDWI")

    # 3. New flood mask
    im2 = axes[2].imshow(new_flood, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"{event_name} - Newly flooded area")
    axes[2].axis("off")

    # 4. Water before
    im3 = axes[3].imshow(water_before, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title(f"{event_name} - Water mask before")
    axes[3].axis("off")

    # 5. Water after
    im4 = axes[4].imshow(water_after, cmap="gray", vmin=0, vmax=1)
    axes[4].set_title(f"{event_name} - Water mask after")
    axes[4].axis("off")

    # 6. NDWI change
    im5 = axes[5].imshow(
        ndwi_change,
        cmap="RdBu",
        vmin=-vmax_change,
        vmax=vmax_change
    )
    axes[5].set_title(f"{event_name} - NDWI change")
    axes[5].axis("off")
    cbar5 = fig.colorbar(im5, ax=axes[5], fraction=0.046, pad=0.04)
    cbar5.set_label(r"$\Delta$NDWI")

    # Optional overall title
    fig.suptitle(
        f"{event_name}: flood detection from Sentinel-2 NDWI and derived masks",
        fontsize=14,
        y=1.02
    )

    return fig


In [ ]:
figure_path = Path(f"{event_to_plot}_flood_report_figure.png")
fig = plot_flood_report_figure(arrays_by_event[event_to_plot], event_to_plot, ndwi_threshold=WATER_NDWI_THRESHOLD)
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to: {figure_path}")
plt.close(fig)


In [ ]:
import rasterio

path = r"../../data/raw/floods/Emilia/after_flooding/2023-05-23-00_00_2023-05-23-23_59_Sentinel-2_L2A_B03_(Raw).tiff"

with rasterio.open(path) as src:
    print("Bounds:", src.bounds)
    print("CRS:", src.crs)
    print("Width, Height:", src.width, src.height)
    print("Transform:", src.transform)